
# Planet → Lake Mask Pipeline (Single Shapefile, **Robust to Invalid Geoms**)

This version is resilient to **invalid polygons** in the lakes shapefile. It attempts to fix invalid
geometries, and if a union operation still fails, it gracefully falls back (or skips) and **continues to the next tile**.


## 0) Requirements

In [ ]:

# If running locally and you need to install packages, uncomment:
# %pip install geopandas rasterio shapely pyproj tqdm numpy fiona


## 1) Imports

In [1]:

from pathlib import Path
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.warp import reproject, Resampling
import geopandas as gpd
from shapely.geometry import box
from shapely.ops import unary_union
from tqdm import tqdm


## 2) Configuration

In [2]:

# --- EDIT THESE PATHS ---
# Folder holding your SR and UDM2 GeoTIFFs (they can be in the same folder)
DATA_DIR = Path(r"D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\chips_out_256")   # change to your folder

# A **single** lakes Shapefile containing polygons for all lakes
LAKES_SHP = Path(r"D:\Thesis\glacial_lake_detection_thesis\data\raw\shapefile\HKH-PK.zip")  # <— change to your lakes .shp path

# Example single-file run (edit to your actual filenames if you want to run only one pair)
SR_TIF = DATA_DIR / "20200906_050946_11_2235_3B_AnalyticMS_SR_8b_clip_reproject_file_format.tif"
UDM2_TIF = None  # set to None if not using UDM2

# Batch mode matching patterns:
SR_MATCH_TOKEN = "_AnalyticMS_SR_8b_"

def sr_to_udm2(sr_path: Path) -> Path:
    """Best-effort mapping from SR filename to UDM2 filename."""
    return sr_path.with_name(sr_path.name.replace("_AnalyticMS_SR_8b_", "_udm2_"))

# Output folder
OUT_DIR = Path(r"D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\labels_out_256")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Label schema
LAKE_VAL = 1
NON_LAKE_VAL = 0
IGNORE_VAL = 255  # used where UDM2 marks pixel as bad

FILL_VALUE = NON_LAKE_VAL

print("Configured paths:")
print("DATA_DIR =", DATA_DIR.resolve())
print("LAKES_SHP =", LAKES_SHP.resolve())
print("OUT_DIR =", OUT_DIR.resolve())


Configured paths:
DATA_DIR = D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\chips_out_256
LAKES_SHP = D:\Thesis\glacial_lake_detection_thesis\data\raw\shapefile\HKH-PK.zip
OUT_DIR = D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\labels_out_256


## 3) Robust geometry helpers

In [3]:

def _try_make_valid(geom):
    """Try multiple strategies to fix invalid geometry; return fixed or None."""
    if geom is None:
        return None
    try:
        if geom.is_empty:
            return None
    except Exception:
        pass
    try:
        if geom.is_valid:
            return geom
    except Exception:
        pass

    # Try shapely.validation.make_valid (Shapely >= 2.0)
    try:
        from shapely.validation import make_valid as sv_make_valid
        fixed = sv_make_valid(geom)
        if fixed and (not fixed.is_empty) and fixed.is_valid:
            return fixed
    except Exception:
        pass

    # Try shapely.make_valid (older alias in some builds)
    try:
        from shapely import make_valid as s_make_valid
        fixed = s_make_valid(geom)
        if fixed and (not fixed.is_empty) and fixed.is_valid:
            return fixed
    except Exception:
        pass

    # Fallback: buffer(0)
    try:
        fixed = geom.buffer(0)
        if fixed and (not fixed.is_empty) and fixed.is_valid:
            return fixed
    except Exception:
        pass

    return None


## 4) Load polygons (robust) from the single lakes Shapefile

In [4]:

def load_lake_polygons_from_single_shapefile(sr_dataset, lakes_shp: Path):
    """
    Load polygons from a single lakes shapefile, fix invalid geoms, reproject to SR CRS,
    filter to those intersecting the SR bounds, and return a geometry list.

    On union errors, fall back to returning the list of fixed polygons (no union),
    and if rasterization still fails later, the caller will catch and skip this tile.
    """
    if not lakes_shp.exists():
        raise FileNotFoundError(f"Lakes shapefile not found: {lakes_shp}")

    gdf = gpd.read_file(lakes_shp)
    if gdf.empty:
        print(f"WARNING: {lakes_shp} is empty.")
        return []
    if gdf.crs is None:
        raise ValueError(f"{lakes_shp} has no CRS defined. Please define it before using this notebook.")

    sr_crs = sr_dataset.crs
    sr_bounds_poly = box(*sr_dataset.bounds)

    # Fix invalid geoms BEFORE reprojection to avoid failures during transform
    gdf['geometry'] = gdf['geometry'].apply(_try_make_valid)
    gdf = gdf[~gdf['geometry'].isna() & ~gdf.geometry.is_empty]
    if gdf.empty:
        return []

    # Reproject to SR CRS and keep polygonal
    gdf_sr = gdf.to_crs(sr_crs)
    gdf_sr = gdf_sr[gdf_sr.geometry.type.isin(["Polygon", "MultiPolygon"])]
    # Intersect with raster bounds
    gdf_sr = gdf_sr[gdf_sr.geometry.intersects(sr_bounds_poly)]
    if gdf_sr.empty:
        return []

    geoms_list = [g for g in gdf_sr.geometry if g is not None and not g.is_empty]
    # Try union to simplify; if it fails, just return list of polygons
    try:
        unioned = unary_union(geoms_list)
        if unioned.is_empty:
            return []
        if unioned.geom_type in ("Polygon", "MultiPolygon"):
            return [unioned]
        return [g for g in getattr(unioned, "geoms", []) if g.geom_type in ("Polygon", "MultiPolygon")]
    except Exception as e:
        print(f"WARNING: unary_union failed due to: {e}. Falling back to non-unioned polygons (tile will still proceed)." )
        return geoms_list


## 5) UDM2 → ignore mask (heuristics)

In [ ]:

def udm2_to_ignore_mask(udm2_path: Path, sr_dataset) -> np.ndarray:
    with rasterio.open(udm2_path) as udm2:
        same_grid = (udm2.transform == sr_dataset.transform) and (udm2.width == sr_dataset.width) and (udm2.height == sr_dataset.height) and (udm2.crs == sr_dataset.crs)
        if same_grid:
            arr = udm2.read()
        else:
            arr = udm2.read()
            out = np.zeros((udm2.count, sr_dataset.height, sr_dataset.width), dtype=arr.dtype)
            for b in range(udm2.count):
                reproject(
                    source=arr[b], destination=out[b],
                    src_transform=udm2.transform, src_crs=udm2.crs,
                    dst_transform=sr_dataset.transform, dst_crs=sr_dataset.crs,
                    resampling=Resampling.nearest
                )
            arr = out

        if arr.shape[0] == 1:
            return (arr[0] == 0)
        clear = arr[0]
        others = arr[1:]
        any_flag = np.any(others > 0, axis=0) if others.size else False
        return (clear == 0) | any_flag


## 6) Build label mask (robust; skip on errors)

In [5]:
import csv

def build_label_mask_for_sr(sr_path: Path, lakes_shp: Path, out_path: Path, udm2_path: Path | None = None,
                            lake_val: int = 1, non_lake_val: int = 0, ignore_val: int = 255):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with rasterio.open(sr_path) as sr:
            lake_polys = load_lake_polygons_from_single_shapefile(sr, lakes_shp)

            # Base mask: non-lake
            mask = np.full((sr.height, sr.width), fill_value=non_lake_val, dtype=np.uint8)

            if lake_polys:
                try:
                    lake_raster = rasterize(
                        [(geom, lake_val) for geom in lake_polys],
                        out_shape=(sr.height, sr.width),
                        transform=sr.transform,
                        fill=non_lake_val,
                        dtype=np.uint8
                    )
                    mask = lake_raster.astype(np.uint8)
                except Exception as e:
                    print(f"WARNING: Rasterize failed for {sr_path.name}: {e}. Skipping this tile.")
                    return  # skip this tile
            #else:
             #   print(f"No lake polygons intersect {sr_path.name}; writing all {non_lake_val}.")

            #if udm2_path is not None and Path(udm2_path).exists():
             #   try:
              #      ignore_mask = udm2_to_ignore_mask(Path(udm2_path), sr)
               #     mask = np.where(ignore_mask, ignore_val, mask).astype(np.uint8)
                #except Exception as e:
                 #   print(f"WARNING: UDM2 processing failed for {sr_path.name}: {e}. Continuing without UDM2.")

            profile = sr.profile.copy()
            profile.update({"count": 1, "dtype": "uint8", "nodata": ignore_val})
            with rasterio.open(out_path, "w", **profile) as dst:
                dst.write(mask, 1)

            #n_ignore = int((mask == ignore_val).sum())
            n_lake = int((mask == lake_val).sum())
            n_nonlake = int((mask == non_lake_val).sum())
            #print(f"Wrote {out_path.name} → shape={mask.shape}, lake={n_lake:,}, nonlake={n_nonlake:,}, ignore={n_ignore:,}")
                    # Append run stats to a CSV (one row per call)
            csv_path = out_path.parent / "label_mask_stats.csv"
            csv_path.parent.mkdir(parents=True, exist_ok=True)
            write_header = not csv_path.exists()
            with csv_path.open("a", newline="") as f:
                writer = csv.writer(f)
                if write_header:
                    writer.writerow([
                        "sr_path", "out_mask", "height", "width",
                        "lake_val", "non_lake_val", "ignore_val",
                        "n_lake", "n_nonlake"
                    ])
                writer.writerow([
                    str(sr_path), out_path.name, mask.shape[0], mask.shape[1],
                    int(lake_val), int(non_lake_val), int(ignore_val),
                    n_lake, n_nonlake
                ])
    except Exception as e:
        print(f"ERROR processing {sr_path.name}: {e}. Skipping this tile.")
        return


## 7) Run → Single tile

In [ ]:

if SR_TIF.exists():
    out_single = OUT_DIR / (SR_TIF.stem + "_label.tif")
    use_udm2 = UDM2_TIF if (UDM2_TIF is not None and UDM2_TIF.exists()) else None
    build_label_mask_for_sr(SR_TIF, LAKES_SHP, out_single, use_udm2,
                            lake_val=LAKE_VAL, non_lake_val=NON_LAKE_VAL, ignore_val=IGNORE_VAL)
else:
    print("Edit SR_TIF/UDM2_TIF/paths above to run the single-tile example.")


## 8) Batch mode → Process all SR tiles in a folder

In [6]:

sr_files = sorted([p for p in DATA_DIR.glob("*.tif")])
print(f"Found {len(sr_files)} SR tiles")

for sr_path in tqdm(sr_files):
    out_label = OUT_DIR / (sr_path.stem + "_label.tif")
    udm2_guess = sr_to_udm2(sr_path)
    use_udm2 = udm2_guess if udm2_guess.exists() else None
    build_label_mask_for_sr(sr_path, LAKES_SHP, out_label, use_udm2,
                            lake_val=LAKE_VAL, non_lake_val=NON_LAKE_VAL, ignore_val=IGNORE_VAL)


Found 6295 SR tiles


100%|██████████| 6295/6295 [41:00<00:00,  2.56it/s]


## 9) (Optional) Build a CSV index

In [ ]:

import csv

index_csv = OUT_DIR / "dataset_index_single_shp_robust.csv"
rows = []
for lbl in sorted(OUT_DIR.glob("*_label.tif")):
    sr_name = lbl.name.replace("_label.tif", ".tif")
    candidate_sr = DATA_DIR / sr_name
    candidate_udm2 = sr_to_udm2(candidate_sr)
    rows.append({
        "sr_path": str(candidate_sr.resolve() if candidate_sr.exists() else ""),
        "udm2_path": str(candidate_udm2.resolve() if candidate_udm2.exists() else ""),
        "label_path": str(lbl.resolve())
    })

with open(index_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["sr_path", "udm2_path", "label_path"])
    w.writeheader()
    w.writerows(rows)

print("Wrote", index_csv.resolve())
